# 🎬 אולפן המשנה — תמונות ניסיון ב־GPU חינמי

**מיועד לברכות א׳, א׳.** זו סביבת ניסיון פרטית ולא פרסום או אישור נוסח. אין שימוש בכלי התמונות או במכסות התמונות של ChatGPT.

1. ב־Colab: **Runtime → Change runtime type → GPU (למשל T4)**. השתמש במכסת ה־Free; אם יש לך מנוי Colab בתשלום, בדוק שלא נצרכים קרדיטים בתשלום.
2. לחץ **Runtime → Run all**. ברירת המחדל היא **שוט SH09 חדש**; לא חוזרים על ניסיון SH01 שכבר הושלם ונדחה. ניתן לבחור 5 תמונות של סצנה אחת, או 16 שוטים מתוכננים.
3. אם כבר ניסית את אותו שוט ורוצה **תמונה שונה**, בחר בהגדרות „ניסיון 2” או „ניסיון 3” (רק עבור איור SH09 בודד). ניסיון 1 חוזר למטמון כשהתמונה תואמת בדיוק — כדי לא לבזבז GPU.
4. המודל מקבל הוראה קצרה וברורה לסצנה והוראה נפרדת לסגנון, ובודק שאין חיתוך של טקסט חיוני.
5. לאחר ההרצה יוצג גיליון סקירה אחד עם כל האיורים שנוצרו, ויופיע קובץ ZIP הכולל אותם בנפרד וגם גיליון סקירה מסומן כטיוטה. אין תוכנות להתקין במחשב שלך.

**אזהרה אמיתית:** GPU חינמי אינו מובטח. אם הוא לא זמין, המחברת עוצרת לפני הורדת מודל. משתמשים רק במשאבים שהשירות מעניק בחינם; אין קוד תשלום או API בתשלום.

**מקורות וזכויות:** המודל SDXL base שוחרר תחת CreativeML Open RAIL++-M; כל האיורים יהיו המחשה אמנותית ולא דיוקן או שחזור מאומת. אין לפרסם בלי ביקורת תורנית, בדיקת זכויות ואישור.


In [ ]:
# @title 🎞️ בחירת כמות התמונות (ללא שימוש ב-ChatGPT Image)
BATCH = "איור אחד חדש - SH09" # @param ["איור אחד חדש - SH09", "חמישה איורים - סצנת הבנים", "כל 16 איורי הניסיון"]
IMAGE_QUALITY = "איכות גבוהה - 1024x576" # @param ["איכות גבוהה - 1024x576", "מהיר - 768x448"]
TAKE = "ניסיון 1 - seed בסיסי" # @param ["ניסיון 1 - seed בסיסי", "ניסיון 2 - זווית חלופית", "ניסיון 3 - זווית חלופית"]
AUTO_DOWNLOAD_ZIP = True # @param {type:"boolean"}
# הפעלה דרך מחברת Google Colab עם GPU חינמי בלבד; אין בקוד שירות תמונות בתשלום.


In [ ]:
import sys, platform
try:
    import torch
except ImportError as exc:
    raise RuntimeError("לא זוהתה סביבת Colab/PyTorch. פתח את המחברת ב-Google Colab.") from exc

if not torch.cuda.is_available():
    raise RuntimeError("אין GPU חינמי מחובר. ב-Colab בחר Runtime > Change runtime type > GPU ואז Run all. לא הופקה תמונה ולא נקרא שירות בתשלום.")

GPU_NAME = torch.cuda.get_device_name(0)
GPU_GIB = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
print("GPU:", GPU_NAME, "VRAM GiB:", GPU_GIB)
if GPU_GIB < 10:
    raise RuntimeError("ה-GPU קטן מדי למודל הניסיון. לא מבוצעת הורדת מודל או יצירת תמונות.")
print("זוהה GPU בחשבון. אי אפשר לקבוע מהקוד אם חשבונך משתמש במכסה חינמית או בקרדיטים; בדוק זאת לפני ההרצה.")


In [ ]:
import json, urllib.request, hashlib
BASE = "https://raw.githubusercontent.com/barkinam21-pixel/project-mishna/main/episodes/berakhot-01-01/"
PROMPT_SOURCE = BASE + "free-cloud-image-prompts.json"
with urllib.request.urlopen(PROMPT_SOURCE, timeout=30) as stream:
    plan = json.loads(stream.read().decode("utf-8"))
assert plan["episode_id"] == "berakhot-01-01"
assert plan["status"] == "private_style_tests_only"
assert plan["monetary_spend_authorized_usd"] == 0
assert plan["public_release_authorized"] is False
assert plan["torah_review_approved"] is False
assert plan["model_id"] == "stabilityai/stable-diffusion-xl-base-1.0"
assert len(plan["shots"]) == 16 and len({s["shot_id"] for s in plan["shots"]}) == 16
for s in plan["shots"]:
    assert s["shot_id"].startswith("SH") and s["generated"] is False and s["accepted"] is False
SELECTED_IDS = {
    "איור אחד חדש - SH09": ["SH09"],
    "חמישה איורים - סצנת הבנים": ["SH09", "SH10", "SH11", "SH12", "SH13"],
    "כל 16 איורי הניסיון": [s["shot_id"] for s in plan["shots"]],
}[BATCH]
selected = [s for s in plan["shots"] if s["shot_id"] in SELECTED_IDS]
assert len(selected) == len(SELECTED_IDS)
# One-click alternatives for SH09 only. Do not multiply GPU work for full batches.
TAKE_INDEX = {
    "ניסיון 1 - seed בסיסי": 0,
    "ניסיון 2 - זווית חלופית": 1,
    "ניסיון 3 - זווית חלופית": 2,
}[TAKE]
if TAKE_INDEX and SELECTED_IDS != ["SH09"]:
    raise RuntimeError("Alternate seed is for a single SH09 preview only; select one image.")

def planned_seed(base_seed, variation_index):
    """Deterministic bounded alternative, never an untracked random rerun."""
    if type(variation_index) is not int or variation_index not in (0, 1, 2):
        raise ValueError("Only three reviewed SH09 seed variants are allowed")
    return int(base_seed) + 1009 * variation_index
if IMAGE_QUALITY == "מהיר - 768x448":
    WIDTH, HEIGHT, STEPS = 768, 448, 16
else:
    WIDTH, HEIGHT, STEPS = 1024, 576, 24
print(f"נבחרו {len(selected)} איורי ניסיון, {WIDTH}x{HEIGHT}, {STEPS} צעדים, ניסיון {TAKE_INDEX+1}. אין התקשרות לספק בתשלום.")


In [ ]:
import subprocess
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "diffusers>=0.35,<1", "transformers>=4.50,<5",
    "accelerate>=1.2,<2", "safetensors>=0.5"
])
print("ספריות חינמיות הותקנו על המכונה של Colab, לא במחשב האישי.")


In [ ]:
import os, time, zipfile, datetime, pathlib
from PIL import Image, ImageDraw, ImageFont, ImageOps
from diffusers import StableDiffusionXLPipeline
from IPython.display import display, FileLink

OUT = pathlib.Path("/content/mishnah-free-image-trials")
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "prompt-pack-snapshot.json").write_text(json.dumps(plan, ensure_ascii=False, indent=2)+"\n", encoding="utf-8")
ZIP_PATH = pathlib.Path("/content/mishnah-free-image-trials.zip")
ZIP_PATH.unlink(missing_ok=True)  # Never offer a stale ZIP after a failed attempt.
LEDGER = OUT / "usage-ledger.jsonl"

def cached_image_matches(path, shot_id, generation_spec_sha256):
    """Reuse only a byte-verified image with a matching complete generation specification."""
    if not path.is_file() or not LEDGER.is_file():
        return False
    actual_sha = hashlib.sha256(path.read_bytes()).hexdigest()
    for line in reversed(LEDGER.read_text(encoding="utf-8").splitlines()):
        try:
            record = json.loads(line)
        except ValueError:
            continue
        if record.get("shot_id") == shot_id and record.get("image_sha256") == actual_sha:
            return record.get("generation_spec_sha256") == generation_spec_sha256
    return False


# Do not download or load a multi-GB image model if every requested image
# is already a verified cache hit. This matters on limited free Colab runtimes.
pipe = None

def load_pipeline_only_when_needed():
    new_pipe = StableDiffusionXLPipeline.from_pretrained(
        plan["model_id"],
        torch_dtype=torch.float16,
        variant="fp16",
        use_safetensors=True
    )
    new_pipe.enable_vae_slicing()
    if GPU_GIB >= 14:
        new_pipe.to("cuda")
    else:
        new_pipe.enable_model_cpu_offload()
    new_pipe.set_progress_bar_config(disable=True)
    return new_pipe


def visual_diagnostics(image):
    """Unapproved numeric color/readability hints for before/after comparisons.

    These cannot count people, identify buildings, assess historical correctness,
    or establish artistic approval. A human must still inspect every shot.
    """
    pixels = list(image.convert("RGB").resize(
        (128, 72), resample=Image.Resampling.BILINEAR
    ).getdata())
    if len(pixels) != 128 * 72:
        raise ValueError("Unexpected diagnostic thumbnail geometry")
    luminance = [(54*r + 183*g + 19*b) // 256 for r, g, b in pixels]
    saturation = [
        (max(r, g, b) - min(r, g, b)) * 255 / max(max(r, g, b), 1)
        for r, g, b in pixels
    ]
    ordered = sorted(luminance)
    hsum = sum(abs(luminance[y*128 + x] - luminance[y*128 + x-1])
               for y in range(72) for x in range(1, 128))
    vsum = sum(abs(luminance[y*128 + x] - luminance[(y-1)*128 + x])
               for y in range(1, 72) for x in range(128))
    p10, p90 = ordered[int((len(ordered)-1)*0.10)], ordered[int((len(ordered)-1)*0.90)]
    return {
        "metric_schema": "draft_visual_v1",
        "saturation_mean_0_255": round(sum(saturation)/len(saturation), 1),
        "luma_p10": p10, "luma_p90": p90, "luma_p90_p10_spread": p90-p10,
        "mean_neighbor_luma_delta": round((hsum+vsum)/(72*127+71*128), 2),
        "meaning": "descriptive_only_not_a_quality_or_scene_approval"
    }


SHEET = OUT / "review-contact-sheet.jpg"
# Do not accidentally ship an earlier batch's review sheet.
SHEET.unlink(missing_ok=True)

def build_contact_sheet(paths):
    """Create a derived, unapproved REVIEW sheet, never a source illustration."""
    if not paths:
        raise ValueError("No real image files to review")
    tile_width, tile_height, gap, header = 520, 332, 16, 78
    columns = 2 if len(paths) > 1 else 1
    rows = (len(paths) + columns - 1) // columns
    canvas = Image.new("RGB", (gap + columns * (tile_width + gap),
                               header + rows * (tile_height + gap)), "#171b27")
    paint = ImageDraw.Draw(canvas)
    try:
        font = ImageFont.truetype("DejaVuSans.ttf", 22)
    except OSError:
        font = ImageFont.load_default()
    paint.text((gap, 12), "DRAFT CONTACT SHEET - NOT APPROVED", fill="#fff2dc", font=font)
    paint.text((gap, 47), "PRIVATE VISUAL REVIEW ONLY - 0 source images approved",
               fill="#b9beca", font=ImageFont.load_default())
    for i, path in enumerate(paths):
        col, row = i % columns, i // columns
        x = gap + col * (tile_width + gap)
        y = header + row * (tile_height + gap)
        paint.rectangle((x, y, x + tile_width, y + tile_height), fill="#272e3b")
        with Image.open(path) as original:
            # 'contain' shows the complete original composition; no crop.
            frame = ImageOps.contain(original.convert("RGB"), (tile_width - 24, 278),
                                     method=Image.Resampling.LANCZOS)
        px = x + (tile_width - frame.width) // 2
        canvas.paste(frame, (px, y + 12 + (278 - frame.height) // 2))
        paint.text((x + 14, y + 295), path.stem + " - UNAPPROVED",
                   fill="#fff2dc", font=font)
    canvas.save(SHEET, format="JPEG", quality=86, optimize=True)
    return SHEET

def save_zip():
    """Export only this run's verified image selection; keep legacy files local.

    A previous Colab session may leave SH01 or stale SHxx images in OUT.
    Including them in a new ZIP would make the intake reject an otherwise
    valid SH09 batch, or consume phone bandwidth for unrelated artwork.
    """
    if not successful:
        raise RuntimeError("No selected images to export")
    wanted = [OUT / "prompt-pack-snapshot.json", LEDGER, *successful]
    if SHEET.is_file():
        wanted.append(SHEET)
    if len({path.name for path in wanted}) != len(wanted):
        raise RuntimeError("Duplicate export path")
    for path in wanted:
        if path.parent != OUT or not path.is_file():
            raise RuntimeError("Missing or unsafe export member: " + path.name)
    tmp = ZIP_PATH.with_suffix(".zip.tmp")
    with zipfile.ZipFile(tmp, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for file in sorted(wanted):
            zf.write(file, arcname=file.name)
    os.replace(tmp, ZIP_PATH)

successful = []
for index, shot in enumerate(selected, 1):
    image_path = OUT / (shot["shot_id"] + ".png")
    # Scene-first instructions avoid burying shot content after a long style string.
    # SDXL has two CLIP text encoders; use one for subject and one for paint style.
    scene_prompt = shot["render_lead"]
    style_prompt = shot.get("render_style", plan["render_style"])
    negative_prompt = shot.get("render_negative_prompt", plan["render_negative_prompt"])
    prompt_digest = hashlib.sha256(scene_prompt.encode("utf-8")).hexdigest()
    prompt_2_digest = hashlib.sha256(style_prompt.encode("utf-8")).hexdigest()
    negative_digest = hashlib.sha256(negative_prompt.encode("utf-8")).hexdigest()
    seed = planned_seed(shot["seed"], TAKE_INDEX)
    guidance = float(shot.get("guidance_scale", 6.0))
    if not (4.0 <= guidance <= 9.0):
        raise RuntimeError("Invalid experimental shot guidance")
    generation_spec = {
        "model_id": plan["model_id"], "model_revision": "unknown",
        "variant": "fp16", "prompt_strategy": "dual_encoder_scene_style_v1",
        "prompt_sha256": prompt_digest,
        "prompt_2_sha256": prompt_2_digest,
        "negative_prompt_sha256": negative_digest,
        "negative_prompt_2_sha256": negative_digest,
        "seed": seed, "variation_index": TAKE_INDEX,
        "width": WIDTH, "height": HEIGHT,
        "inference_steps": STEPS, "guidance_scale": guidance
    }
    spec_hash = hashlib.sha256(json.dumps(generation_spec, sort_keys=True).encode("utf-8")).hexdigest()
    # A same-named PNG alone is not evidence that its prompt/model/settings match.
    if cached_image_matches(image_path, shot["shot_id"], spec_hash):
        print("מטמון מאומת SHA+מפרט:", image_path.name)
        successful.append(image_path)
        continue
    if image_path.exists():
        old_hash = hashlib.sha256(image_path.read_bytes()).hexdigest()
        archived = OUT / (shot["shot_id"] + ".superseded-" + old_hash[:12] + ".png")
        if archived.exists() and hashlib.sha256(archived.read_bytes()).hexdigest() != old_hash:
            raise RuntimeError("Archive name collision; refusing to overwrite an old image")
        image_path.replace(archived)
        print("קובץ ישן נשמר בנפרד, לא ישמש בטעות כמטמון:", archived.name)
    started = time.monotonic()
    print(f"[{index}/{len(selected)}] יוצר {shot['shot_id']} על GPU של החשבון; ללא API בתשלום")
    try:
        if pipe is None:
            pipe = load_pipeline_only_when_needed()
        # Prevent silent CLIP truncation before any GPU inference.
        for tokenizer, label, content in (
            (pipe.tokenizer, "scene", scene_prompt),
            (pipe.tokenizer_2, "style", style_prompt),
            (pipe.tokenizer, "negative scene", negative_prompt),
            (pipe.tokenizer_2, "negative style", negative_prompt)
        ):
            token_ids = tokenizer(content, truncation=False)["input_ids"]
            if len(token_ids) > tokenizer.model_max_length:
                raise RuntimeError(f"{label} exceeds CLIP token context: {len(token_ids)}")
        generator = torch.Generator(device="cuda").manual_seed(seed)
        result = pipe(
            prompt=scene_prompt,
            prompt_2=style_prompt,
            negative_prompt=negative_prompt,
            negative_prompt_2=negative_prompt,
            width=WIDTH,
            height=HEIGHT,
            num_inference_steps=STEPS,
            guidance_scale=guidance,
            generator=generator
        )
        image = result.images[0]
        visual_metrics = visual_diagnostics(image)
        image.save(image_path, format="PNG")
        sha = hashlib.sha256(image_path.read_bytes()).hexdigest()
        record = {
            "date_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
            "shot_id": shot["shot_id"], "scene_id": shot["scene_id"],
            "model_id": plan["model_id"], "model_revision": "unknown",
            "model_license": plan["model_license"],
            "seed": seed, "variation_index": TAKE_INDEX,
            "prompt_sha256": prompt_digest,
            "prompt_2_sha256": prompt_2_digest,
            "negative_prompt_sha256": negative_digest,
            "negative_prompt_2_sha256": negative_digest,
            "generation_spec_sha256": spec_hash, "generation_spec": generation_spec,
            "image_sha256": sha, "image_size": [WIDTH, HEIGHT],
            "inference_steps": STEPS, "gpu": GPU_NAME,
            "visual_diagnostics": visual_metrics,
            "duration_seconds": round(time.monotonic()-started, 1),
            "operation": "image_generate", "accepted": False,
            "public_release_authorized": False,
            "paid_api_calls": 0, "external_cash_spend_initiated_usd": 0,
            "colab_gpu_quota_usage": "unknown",
            "chatgpt_image_quota_usage": 0
        }
        with LEDGER.open("a", encoding="utf-8") as fh:
            fh.write(json.dumps(record, ensure_ascii=False)+"\n")
        successful.append(image_path)
        build_contact_sheet(successful)
        save_zip()  # Preserve progress after each generated image.
        print("נשמר:", image_path.name, "| SHA-256:", sha[:12])
        print("מדדי צבע ובהירות להשוואת ניסויים בלבד (לא אישור אומנותי):", visual_metrics)
    except torch.cuda.OutOfMemoryError:
        print("GPU memory insufficient. Earlier images remain in ZIP if any.")
        torch.cuda.empty_cache()
        break
    except KeyboardInterrupt:
        print("עצירה ידנית; תמונות קודמות נשמרו ב-ZIP.")
        break
    except Exception as exc:
        print("עצרתי בלי לעקוף כשל או להפעיל שירות אחר:", type(exc).__name__, str(exc)[:150])
        break

if successful:
    build_contact_sheet(successful)
    save_zip()
    print(f"נוצרו {len(successful)} תמונות בסשן. כולן טיוטות לאישור אנושי.")
    display(Image.open(SHEET).copy())
    print("קישור ZIP להורדה:")
    display(FileLink(str(ZIP_PATH)))
    if AUTO_DOWNLOAD_ZIP:
        try:
            from google.colab import files
            files.download(str(ZIP_PATH))
        except ImportError:
            print("אפשר להוריד מקישור ה-ZIP שמוצג למעלה.")
else:
    print("לא נוצרה תמונה; אין שימוש ב-ChatGPT Images או שירות חיצוני בתשלום.")
